### Load data

In [1]:
import json

# load 1,000 pubmedqa questions (use json to keep abstract and ID attached)
with open("pubmedqa/data/ori_pqal.json") as f:
    data = json.load(f)

print(len(data))

1000


### Line up questions with their abstracts

In [2]:
# line up each question with its matching abstract and abstract id
abstract_ids = []
abstracts = []
questions = []

for abstract_id, entry in data.items():
    abstract_ids.append(abstract_id)
    abstracts.append(" ".join(entry["CONTEXTS"]))
    questions.append(entry["QUESTION"])

print(len(abstracts))

1000


### Sparse search (BM25)

In [3]:
import bm25s

# split each abstract into words (also drops filler words)
abstract_tokens = bm25s.tokenize(abstracts, stopwords="en")

# create keyword search index for all abstracts
bm25 = bm25s.BM25()
bm25.index(abstract_tokens)

/Users/andrew/Documents/Berkeley/MIDS/DS266/Final Project/DS266_RAG/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# try one question to see what bm25 finds
question = questions[0]
question_tokens = bm25s.tokenize(question, stopwords="en")

# retrieve top 5 results
top_positions, top_scores = bm25.retrieve(question_tokens, k=5)

print("question:", question)
print("correct abstract is at position: 0")
print("bm25's top 5 positions:", top_positions[0])
print("bm25's scores:", top_scores[0])

question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
correct abstract is at position: 0
bm25's top 5 positions: [  0 265 111 244 450]
bm25's scores: [21.529465   9.112523   5.512696   4.3754864  4.285845 ]


### Dense search (PubMedBERT)

In [5]:
from txtai import Embeddings

# turn every abstract into numbers that capture its meaning
dense = Embeddings(path="neuml/pubmedbert-base-embeddings")
dense.index(abstracts)

W1105 12:58:08.072000 94200 .venv/lib/python3.14/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6394.00it/s]


In [6]:
# try the same question with dense search
dense_results = dense.search(question, 5)

print("question:", question)
print("correct abstract is at position: 0")
print("dense's top 5 results:", dense_results)

question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
correct abstract is at position: 0
dense's top 5 results: [(0, 0.6894745230674744), (265, 0.3121076822280884), (76, 0.18515199422836304), (199, 0.16055384278297424), (751, 0.15123626589775085)]


## Score search on all 1,000 questions

In [7]:
# % of questions where the correct abstract shows up in the top k results
def recall_at_k(all_results, k):
    found = 0
    for question_position, result_positions in enumerate(all_results):
        if question_position in result_positions[:k]:
            found += 1
    return found / len(all_results)

In [8]:
# search all 1,000 questions with bm25 and keep the top 10 for each
all_question_tokens = bm25s.tokenize(questions, stopwords="en")
bm25_results, bm25_scores = bm25.retrieve(all_question_tokens, k=10)

print("bm25 recall@1:", recall_at_k(bm25_results, 1))
print("bm25 recall@5:", recall_at_k(bm25_results, 5))
print("bm25 recall@10:", recall_at_k(bm25_results, 10))

bm25 recall@1: 0.947
bm25 recall@5: 0.983
bm25 recall@10: 0.986


In [9]:
# search all 1,000 questions with dense and keep the top 10 for each
dense_raw_results = dense.batchsearch(questions, 10)

# keep just the positions so it matches the bm25 results
dense_results = []
for question_results in dense_raw_results:
    positions = []
    for position, score in question_results:
        positions.append(position)
    dense_results.append(positions)

print("dense recall@1:", recall_at_k(dense_results, 1))
print("dense recall@5:", recall_at_k(dense_results, 5))
print("dense recall@10:", recall_at_k(dense_results, 10))

dense recall@1: 0.98
dense recall@5: 0.994
dense recall@10: 0.995


## Combine bm25 and dense with RRF

In [11]:
# how many points an abstract gets for its rank in a list
def points_for_rank(rank):
    return 1 / (60 + rank)


# go down one search method's top results and give each abstract points
def add_points(scoreboard, top_abstracts):
    rank = 1

    for abstract_position in top_abstracts:
        # bm25 gives numpy numbers, dense gives normal ones, so make them match
        abstract_position = int(abstract_position)

        # first time seeing this abstract, start it at 0 points
        if abstract_position not in scoreboard:
            scoreboard[abstract_position] = 0

        scoreboard[abstract_position] += points_for_rank(rank)
        rank += 1

# combine bm25 and dense into one ranked list for a question
def rrf_combine(bm25_top_abstracts, dense_top_abstracts):
    scoreboard = {}

    add_points(scoreboard, bm25_top_abstracts)
    add_points(scoreboard, dense_top_abstracts)

    # sort abstracts from most points to least
    combined_ranking = sorted(scoreboard, key=scoreboard.get, reverse=True)
    return combined_ranking

In [12]:
# combine bm25 and dense results for every question
rrf_results = []

for question_position in range(len(questions)):
    bm25_top_abstracts = bm25_results[question_position]
    dense_top_abstracts = dense_results[question_position]

    combined_ranking = rrf_combine(bm25_top_abstracts, dense_top_abstracts)
    rrf_results.append(combined_ranking)

print("rrf recall@1:", recall_at_k(rrf_results, 1))
print("rrf recall@5:", recall_at_k(rrf_results, 5))
print("rrf recall@10:", recall_at_k(rrf_results, 10))

rrf recall@1: 0.966
rrf recall@5: 0.994
rrf recall@10: 0.997


## Results

Recall on all 1,000 PubMedQA questions (% of questions where the correct abstract is in the top k)

| Method | Recall@1 | Recall@5 | Recall@10 |
|---|---|---|---|
| BM25 | 94.7% | 98.3% | 98.6% |
| Dense | 98.0% | 99.4% | 99.5% |
| RRF | 96.6% | 99.4% | 99.7% |